# Week 92, Day 6: Retrospective

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w092d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
planned, done = [18, 16, 15], [13, 15, 15]
print([round(100 * d / p) for p, d in zip(planned, done)])

**Predict before you run:** what will this print?



In [ ]:
estimates = [("S3", 5, 8), ("S5", 3, 3), ("S7", 8, 6)]
print([round(actual / est, 2) for _, est, actual in estimates])

**Predict before you run:** what will this print?



In [ ]:
velocity = [13, 15, 15]
first, last = velocity[0], velocity[-1]
print("up" if last > first * 1.1 else "down" if last < first * 0.9 else "flat")

**Predict before you run:** what will this print?



In [ ]:
notes = [("well", "demo every sprint"), ("badly", "testing users late"), ("try", "test users in sprint 1")]
print({k: sum(1 for c, _ in notes if c == k) for k in ("well", "badly", "learned", "try")})

**Predict before you run:** what will this print?



In [ ]:
import re
print(re.sub(r"[^a-z ]", "", "Tests written LATE!".lower()))

### Exercise: Sorting release tags

Write `sort_tags(tags)`: return the tags that are valid versions (`MAJOR.MINOR.PATCH`, perhaps with a leading `v`, each part a whole number without leading zeros except `0`), sorted from oldest to newest as numbers; ignore anything else.

In [ ]:
import re

def sort_tags(tags):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sort():\n    \"numbers, not text\"\n    assert sort_tags([\"v0.10.0\", \"v0.9.0\", \"v1.0.0\", \"nightly\", \"v0.9.10\", \"0.2.1\", \"v01.0.0\"]) == [\"0.2.1\", \"v0.9.0\", \"v0.9.10\", \"v0.10.0\", \"v1.0.0\"]\n    assert sort_tags([]) == []")

### Exercise: How long are the captions?

Write `caption_span(srt)`: from SRT text, return the seconds from the first caption's start to the last caption's end, rounded to 3 decimal places (`0.0` if there are no captions). Times are `HH:MM:SS,mmm`.

In [ ]:
import re

def caption_span(srt):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SRT = \"1\\n00:00:01,250 --> 00:00:03,500\\nWaymark plans walks\\n\\n2\\n00:01:02,000 --> 00:01:04,125\\nwith honest times.\\n\"\n\ndef test_span():\n    \"first start to last end\"\n    assert caption_span(SRT) == 62.875 and caption_span(\"\") == 0.0")

### Exercise: The retrospective report

Write `retrospective(sprints, notes)`. Each sprint is a dict with `name`, `planned` and `done` (points), `carried` (a list of story ids not finished) and `estimates` (a list of `(story, estimate, actual)`). Each note is `(column, text)` with columns `"well"`, `"badly"`, `"learned"` and `"try"`. Return a list of lines:

1. per sprint: `"NAME: D of P points (R%)"` (R rounded), then, if any, `" carried: S5, S9"` added to the same line;
2. `"velocity: mean M, TREND"`: M the mean of `done` to 1 decimal place; TREND `"up"` if the last sprint's `done` is more than 10% above the first's, `"down"` if more than 10% below, otherwise `"flat"`;
3. `"estimates: N stories, mean ratio X, most underestimated STORY (Y)"`: ratios are actual over estimate, X their mean and Y the largest ratio, both to 2 decimal places (first story wins ties); or `"estimates: none"`;
4. `"notes: W well, B badly, L learned, T try"`;
5. `"actions:"` and then one line `"  - TEXT"` per `try` note, in order, or `"actions: none"` if there are none.

In [ ]:
def retrospective(sprints, notes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SPRINTS = [\n    {\"name\": \"Sprint 1\", \"planned\": 18, \"done\": 13, \"carried\": [\"S5\"], \"estimates\": [(\"S3\", 5, 8), (\"S6\", 3, 3), (\"S8\", 5, 4)]},\n    {\"name\": \"Sprint 2\", \"planned\": 16, \"done\": 15, \"carried\": [], \"estimates\": [(\"S5\", 3, 3), (\"S7\", 8, 10), (\"S4\", 5, 4)]},\n    {\"name\": \"Sprint 3\", \"planned\": 15, \"done\": 15, \"carried\": [], \"estimates\": [(\"S11\", 5, 6), (\"S12\", 8, 8)]},\n]\nNOTES = [(\"well\", \"a demo every sprint kept us honest\"), (\"badly\", \"user testing came too late\"),\n         (\"learned\", \"measure before optimising\"), (\"try\", \"test with users in sprint 1\"),\n         (\"well\", \"conventional commits wrote the release notes\"), (\"badly\", \"estimates for searching were low\"),\n         (\"try\", \"split any story over 5 points\")]\n\ndef test_waymark():\n    \"six weeks of Waymark\"\n    assert retrospective(SPRINTS, NOTES) == [\n        \"Sprint 1: 13 of 18 points (72%) carried: S5\",\n        \"Sprint 2: 15 of 16 points (94%)\",\n        \"Sprint 3: 15 of 15 points (100%)\",\n        \"velocity: mean 14.3, up\",\n        \"estimates: 8 stories, mean ratio 1.08, most underestimated S3 (1.60)\",\n        \"notes: 2 well, 2 badly, 1 learned, 2 try\",\n        \"actions:\",\n        \"  - test with users in sprint 1\",\n        \"  - split any story over 5 points\",\n    ]\n\ndef test_quiet():\n    \"one sprint, no estimates, no actions\"\n    sprint = {\"name\": \"Only\", \"planned\": 10, \"done\": 8, \"carried\": [\"S1\", \"S2\"], \"estimates\": []}\n    assert retrospective([sprint], [(\"well\", \"fine\")]) == [\n        \"Only: 8 of 10 points (80%) carried: S1, S2\", \"velocity: mean 8.0, flat\", \"estimates: none\",\n        \"notes: 1 well, 0 badly, 0 learned, 0 try\", \"actions: none\"]\n    falling = [dict(sprint, done=10, carried=[]), dict(sprint, done=8, carried=[])]\n    assert retrospective(falling, [])[2] == \"velocity: mean 9.0, down\"")

> **Six weeks of Waymark.** Waymark's velocity rose from 13 to 15 points, mostly because stories got smaller and better understood, not because anyone worked harder. The worst estimate was S3, saved routes, the first story with a database migration: the first time is always slower. User testing found the most important problem (the round-walk form) in Week 91, when it would have cost an hour to fix in Week 88. What would your own records say? Write your four columns before reading the bonus.

### Exercise: Bonus: actions and recurring problems

Write `follow_through(actions, retros)`. `actions` is a list of dicts with `action`, `owner` and `done` (a bool) from earlier retrospectives; `retros` is a list of retrospectives, each a list of its "badly" notes. Return a list of lines: `"actions: D of N done (P%)"` (P rounded; `"actions: none"` if there are none); `"  open: OWNER: ACTION"` for each action not done, in order; then, for each problem that appears in two or more retrospectives (compared after lower-casing and keeping only letters and spaces, with spaces collapsed), `"recurring in K retros: TEXT"`, where TEXT is the normalised form, in order of first appearance.

In [ ]:
import re

def follow_through(actions, retros):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("ACTIONS = [{\"action\": \"split stories over 5 points\", \"owner\": \"ada\", \"done\": True},\n           {\"action\": \"test with users earlier\", \"owner\": \"ada\", \"done\": False},\n           {\"action\": \"add a benchmark to CI\", \"owner\": \"bo\", \"done\": True},\n           {\"action\": \"write ADRs as we go\", \"owner\": \"bo\", \"done\": False}]\nRETROS = [[\"Estimates were low\", \"Tests written late!\"], [\"tests written  LATE\", \"CI was slow\"],\n          [\"Estimates were low.\", \"CI was slow\", \"Tests written late\"]]\n\ndef test_follow_through():\n    \"were the actions done?\"\n    assert follow_through(ACTIONS, RETROS) == [\n        \"actions: 2 of 4 done (50%)\",\n        \"  open: ada: test with users earlier\",\n        \"  open: bo: write ADRs as we go\",\n        \"recurring in 2 retros: estimates were low\",\n        \"recurring in 3 retros: tests written late\",\n        \"recurring in 2 retros: ci was slow\",\n    ]\n\ndef test_nothing():\n    \"a first retrospective\"\n    assert follow_through([], [[\"one problem\", \"one problem\"]]) == [\"actions: none\"]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Looking back usefully

A retrospective works when it's **safe** (it examines the process, not people: Norm Kerth's "prime directive" asks everyone to assume each person did the best they could with what they knew at the time), **specific** (evidence, not impressions), and **short on actions** (two or three that will actually happen, each with an owner).

A common format collects notes in four columns: what went **well**, what went **badly**, what we **learned**, and what we'll **try** next. The numbers come from the sprint records: **velocity** (points done per sprint) and its trend, how many stories were **carried over**, and how far **estimates** were from the actual effort (the ratio of actual to estimate: above 1 means it took longer than thought). For a solo capstone, you're the whole team; write the notes anyway.

## Bonus: following through

Retrospectives fail quietly when their actions are never done, and the same problems come back sprint after sprint. Tracking both takes a few lines.

## Weekly quiz

## Week 92 review

Tick off each one honestly. If one feels shaky, go back to that day.

- I can version with semver, work out releases and notes from conventional commits, check `pyproject.toml`, and publish checksums. ([Day 1](w092d1.html))
- I can structure a write-up, measure its sentences, find unsupported claims and record decisions. ([Day 2](w092d2.html))
- I can plan and time a demo, write and check SRT captions, and make chapters. ([Day 3](w092d3.html))
- I can generate an escaped portfolio page and check it for heading, image, link and page problems. ([Day 4](w092d4.html))
- I can plan a talk, check slides, share minutes exactly, and measure rehearsals, pace and fillers. ([Day 5](w092d5.html))
- I can run a retrospective from evidence and check that its actions get done. (Today)

## Your capstone today

1. **Your retrospective**: the four columns, then the numbers from your sprint records through your own `retrospective`.
2. **Two or three actions** for your next project, each with an owner (you) and a way to tell they happened.
3. **Finish the links**: release, write-up, demo, portfolio page and talk slides, all reachable from the README.
4. **Celebrate**: you designed, built, hardened and shipped a real project. Tell someone about it.

That's the end of the capstone. Tomorrow is a **rest day**, and next week is a **rest week**: rest properly. After it, Phase 7 continues with **Open source I**: reading large codebases, how open-source projects work, and your first steps towards contributing to CPython itself.